# Inference Visualizer
## Developed by Moose Abou-Harb on behalf of Paccar Inc

In [30]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

import json
import pathlib
import math

store_folder_path = pathlib.Path.cwd().parent / "test_data"
gt_file = store_folder_path / "all_objects_ground_truth.json"

earth_radius = 6378137.0
deg_to_rad = math.pi / 180

modalities = ["camera", "lidar", "radar"]

def main():
    #Load up the data from the sim
    gt_data = {}
    inferences = {}
    try:
        gt_data = try_load_json(gt_file)
        for modality in modalities:
            modality_path = store_folder_path / f"{modality}_sim_results.json"
            inferences[modality] = try_load_json(modality_path)["inferences"]
    except Exception as e:
        print("Failed to load sim data!")
        return

    #Get the origin
    origin = gt_data["start_pos"]

    #Shove that data into a dataframe
    main_df = pd.DataFrame()
    for modality in modalities:
        new_df = pd.DataFrame(inferences[modality])
        new_df["modality"] = modality
        main_df = pd.concat([main_df, new_df])

    #Expand lat/long/alt into local coords
    main_df[["x", "y", "z"]] = main_df.apply(
        lambda row : geo_to_local(origin, [row["latitude"], row["longitude"], row["altitude"]]), 
        axis=1,
        result_type="expand"
    )

    #Create an origin dataframe
    origin_df = pd.DataFrame([{
        "timestamp" : 0,
        "class" : "origin",
        "latitude" : origin[0],
        "longitude" : origin[1],
        "altitude" : origin[2],
        "dimensions" : [0, 0, 0],
        "obj_id" : -1,
        "modality" : "origin",
        "x" : 0,
        "y" : 0,
        "z" : 0
    }])

    #Merge in the origin point
    main_df = pd.concat([origin_df, main_df])

    #Expand dimensions into individual columns
    main_df[['dx', 'dy', 'dz']] = pd.DataFrame(main_df['dimensions'].tolist(), index=main_df.index)
    main_df = main_df.drop("dimensions", axis=1)
    
    display(main_df.head(30))

    fig = px.scatter_3d(main_df, x='x', y='y', z='z', opacity=0.7, color="modality")

    fig.update_traces(marker=dict(size=5))

    fig.show()

def try_load_json(fp: str) -> dict:
    try:
        with open(fp, 'r') as file:
            data = json.load(file)
            return data
    except FileNotFoundError:
        print(f"Failed to load file: {fp}")
    except json.JSONDecodeError:
        print(f"Selected file contains illegal JSON: {fp}")
    except Exception as e:
        print(f"Something went wrong while loading: {fp}, {e}")

def geo_to_local(origin, target):
    d_lat = target[0] - origin[0]
    d_lon = target[1] - origin[1]
    d_alt = target[2] - origin[2]
    y_meters = d_lat * deg_to_rad * earth_radius

    radius_at_lat = earth_radius * math.cos(origin[0] * deg_to_rad)
    x_meters = d_lon * deg_to_rad * radius_at_lat

    return [x_meters, y_meters, d_alt]

if __name__ == "__main__":
    main()

,timestamp,class,latitude,longitude,altitude,obj_id,modality,x,y,z,dx,dy,dz
0,0.000000,origin,54.557931,25.894047,651.915188,-1,origin,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
0,43.472233,pedestrian,54.557873,25.894293,647.444880,0,camera,15.893600,-6.468479,-4.470309,0.328019,0.417421,0.487318
1,43.452161,vehicle,54.557823,25.893973,654.188750,1,camera,-4.727300,-12.039411,2.273562,0.378786,0.168580,0.492122
2,43.403596,pedestrian,54.557984,25.893764,651.283101,2,camera,-18.246937,5.866208,-0.632088,0.302304,0.786723,0.422097
3,43.458875,vehicle,54.557997,25.894005,656.441320,3,camera,-2.690159,7.397304,4.526132,0.153434,0.409976,0.579554
4,43.409469,vehicle,54.557919,25.893795,649.862357,4,camera,-16.249445,-1.281691,-2.052831,0.175105,0.288237,0.670752
0,43.451100,pedestrian,54.557873,25.894295,647.475847,0,lidar,16.040830,-6.436839,-4.439341,0.305534,0.441792,0.459975
1,43.410917,traffic_cone,54.557982,25.893763,651.204606,2,lidar,-18.326181,5.672083,-0.710582,0.305105,0.779510,0.430662
2,43.499091,vehicle,54.557919,25.893797,649.891910,4,lidar,-16.092058,-1.305474,-2.023279,0.125301,0.313979,0.679756
0,43.443121,traffic_cone,54.557871,25.894285,647.455832,0,radar,15.413374,-6.610893,-4.459357,0.456656,0.362459,0.392943
